In [1]:
import time

start_time = time.perf_counter()

# IMPORT API KEY

In [2]:
import getpass
import os

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Tavily API key:\n")

# SEARCH TOOL DEFINITION

In [3]:
from typing import List, Literal, Optional
from langchain.tools import tool
from langchain_tavily import TavilySearch

@tool
def tavily_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_answer: bool = False,
    include_raw_content: bool = False,
    include_images: bool = False,
    include_image_descriptions: bool = False,
    search_depth: Literal["basic", "advanced"] = "basic",
    time_range: str = "",            # instead of None
    include_domains: List[str] = [], # instead of None
    exclude_domains: List[str] = [], # instead of None
    country: str = ""                # instead of None
) -> str:
    """
    Perform a web search using TavilySearch.

    Args:
        query: The search query string.
        max_results: Maximum number of search results to return.
        topic: Category of the search ("general", "news", "finance").
        include_answer: Include an answer to the original query in results.
        include_raw_content: Include cleaned and parsed HTML of each search result.
        include_images: Include a list of query-related images in the response.
        include_image_descriptions: Include descriptive text for each image.
        search_depth: Depth of the search ("basic" or "advanced").
        time_range: Time range to filter results ("day", "week", "month", "year") or "".
        include_domains: List of domains to specifically include.
        exclude_domains: List of domains to specifically exclude.
        country: Country to focus the search on.

    Returns:
        Search results as a string.
    """
    # parameters validation
    if len(time_range) == 0:
        time_range = None
    if len(include_domains) == 0:
        include_domains = None
    if len(exclude_domains) == 0:
        exclude_domains = None
    if len(country) == 0:
        country = None

    tool_instance = TavilySearch(
        max_results=max_results,
        topic=topic,
        include_answer=include_answer,
        include_raw_content=include_raw_content,
        include_images=include_images,
        include_image_descriptions=include_image_descriptions,
        search_depth=search_depth,
        time_range=time_range,
        include_domains=include_domains,
        exclude_domains=exclude_domains,
        country=country
    )

    return tool_instance.invoke({"query": query})

# EXTRACT TOOL DEFINITION

In [4]:
from typing import Literal
from langchain.tools import tool
from langchain_tavily import TavilyExtract

@tool
def tavily_extract(
    url: list,
    extract_depth: Literal["basic", "advanced"] = "basic",
    chunks_per_source: Literal[1, 2, 3, 4, 5] = 3,
    include_images: bool = False,
    include_favicon: bool = False,
    format_text: Literal["markdown", "text"] = "markdown"
) -> str:
    """
    Extract content from a web page using TavilyExtract.

    Args:
        url: The URL of the pages to extract content from.
        extract_depth: Extraction depth ("basic" or "advanced").
        chunks_per_source: Number of chunks per source (1-5).
        include_images: Whether to include images (True or False).
        include_favicon: Whether to include favicon (True or False).
        format_text: Output format ("markdown" or "text").

    Returns:
        Extracted content as a string.
    """

    tool_instance = TavilyExtract(
        extract_depth=extract_depth,
        chunks_per_source=chunks_per_source,
        include_images=include_images,
        include_favicon=include_favicon,
        format=format_text,
    )

    return tool_instance.invoke({"urls": url})

# RETRIEVE TOOL DEFINITION

In [5]:
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

# AGENT DEFINITION

In [6]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:9b")

agent = create_agent(
    model=model,
    tools=[tavily_search, tavily_extract, retrieve_context],
    checkpointer=InMemorySaver(),
    # system_prompt = """
    #     You are a web research assistant. You can execute the 'tavily_search' tool to perform a web search, and the 
    #     'tavily_extract' tool to perform the extraction of the webpage.

    #     The 'tavily_search' tool must be executed according to these rules:
    #     - The user's input will be in two parts: 'text' (query) and 'params' (dictionary of tool parameters).
    #     - The user will always provide a search query and a dictionary of parameters for the tool.
    #     - Only use the parameters provided by the user.
    #     - Execute the tool using these parameters.

    #     Make sure to use the 'tavily_extract' tool with the following parameters:
    #     - extract_depth = "basic"
    #     - chunks_per_source = 3
    #     - include_images = False
    #     - include_favicon = False
    #     - format_text = "markdown"
    #     Retrieve the entire original content from the user defined webpage. 
    #     Do NOT modify it at all.
    #     """,
    # system_prompt = """
    #     You are a web research assistant. You can use the 'tavily_search' tool to perform web searches, and the 'tavily_extract' 
    #     tool to extract webpage content. 
    #     The user will ask or for a web search or for a webpage extraction. If the user provides a query with some parameters, use 
    #     the 'tavily_search' tool. If the user provides a list of url, use the 'tavily_extract' tool. 
    
    #     Rules for using the 'tavily_search' tool:
    #     The user's input will consist of two parts: 'text' (the query) and 'params' (a dictionary of tool parameters).
    #     The user will always provide a search query and a set of parameters for the tool.
    #     Only use the parameters provided by the user.
    #     Execute the tool using exactly the parameters given.
        
    #     Guidelines for using the 'tavily_extract' tool:
    #     Retrieve the entire original content from the user-defined webpage. The extraction must be an exact replica of the tool 
    #     results without any modifications (no changes to the lexicon, punctuation, format, or language). The output must be 
    #     identical to the tool's result, with no adjustments or alterations whatsoever.
    #     Please strictly follow the rules below:
    #     Set extract_depth to "basic".
    #     Set chunks_per_source to 3.
    #     Set include_images to False.
    #     Set include_favicon to False.
    #     Set format_text to "markdown".
    #     """,
    # system_prompt = """
    # You are a web research assistant. You can use the 'tavily_search' tool to perform web searches, and the 'tavily_extract' 
    # tool to extract webpage content. 
    # The user will ask or for a web search or for a webpage extraction. If the user provides a query with some parameters, use 
    # the 'tavily_search' tool. If the user provides a list of url, use the 'tavily_extract' tool. 
    # Rules for using the 'tavily_search' tool:
    # The user's input will consist of two parts: 'text' (the query) and 'params' (a dictionary of tool parameters).
    # The user will always provide a search query and a set of parameters for the tool.
    # Only use the parameters provided by the user.
    # Execute the tool using exactly the parameters given.
    
    # Guidelines for using the 'tavily_extract' tool:
    # Retrieve the entire original content from the user-defined webpage. The extraction must be an exact replica of the tool 
    # results without any modifications (no changes to the lexicon, punctuation, format, or language). The output must be 
    # identical to the tool's result, with no adjustments or alterations whatsoever.
    
    # REPORTING RULES (CRITICAL) ONLY for the 'tavily_extract' tool:
    # - Your final response MUST be a verbatim report of the extracted content.
    # - DO NOT modify the content at all (no changes to lexicon, punctuation, format, or language).
    # - DO NOT summarize the content.
    # - DO NOT add any introductory or concluding text (e.g., "Here are the extracted recipes").
    # - DO NOT add any AI-generated headers or markdown structures not present in the original tool output.
    # - Your output must consist ONLY of the SOURCE headers and the verbatim content from the tool.
    # - For EACH source in the tool result, format it EXACTLY like this:
        
    #     ----- SOURCE: <url>
    #     <raw_content>
        
    # - Separate different sources with TWO NEWLINES. Example of two results retrieved from the <tavily_extract> tool:

    #     ----- SOURCE: <url_1>
    #     <raw_content_1>


    #     ----- SOURCE: <url_2>
    #     <raw_content_2>
    
    # Please strictly follow the tool parameters below:
    # Set extract_depth to "basic".
    # Set chunks_per_source to 3.
    # Set include_images to False.
    # Set include_favicon to False.
    # Set format_text to "markdown".
    # """,
    system_prompt = """
    You are an advanced web research and conversational RAG assistant. You have three specific tools at your disposal. 
    You must choose the appropriate one based on the user's request.
    
    1. WEB SEARCH: 
    If the user provides a query with some parameters, use the 'tavily_search' tool.
    - The user's input will consist of two parts: 'text' (the query) and 'params' (a dictionary of tool parameters).
    - Only use the parameters provided by the user. Execute the tool using exactly the parameters given.
    
    2. WEBPAGE EXTRACTION: 
    If the user provides a list of URLs, use the 'tavily_extract' tool.
    - Retrieve the entire original content from the webpage without any modifications, adjustments or alterations whatsoever 
    (no changes to the lexicon, punctuation, format, or language).
    - REPORTING RULES (CRITICAL): Your final response MUST be a verbatim report of the extracted content. 
    Do NOT modify the content, summarize it, or add introductory/concluding text.
    - DO NOT add any AI-generated headers or markdown structures not present in the original tool output.
    - Format each source in the 'tavily_extract' tool result exactly like this:
        ----- SOURCE: <url>
        <raw_content>
    - Separate different sources with TWO NEWLINES. Example of two results retrieved from the <tavily_extract> tool:

        ----- SOURCE: <url_1>
        <raw_content_1>


        ----- SOURCE: <url_2>
        <raw_content_2>
        
    - Strictly use these parameters: extract_depth="basic", chunks_per_source=3, include_images=False, include_favicon=False, 
    format_text="markdown".
    
    3. CONVERSATIONAL RAG & QUESTION ANSWERING: 
    If the user asks a natural language question or wants to discuss a specific topic, use the 'retrieve_context' tool.
    - Query the vector store to retrieve relevant context.
    - CRITICAL RULE: You must base your answer *EXCLUSIVELY* on the retrieved context.
    - If the answer to the user's question is not present in the retrieved context, state clearly that you do not have enough 
    information in the extracted sources. DO NOT guess, synthesize outside knowledge, or hallucinate.
    - Maintain a helpful, clear, coincise and conversational tone while answering in the user's language.
    """,
)

# WEB SEARCH INVOCATION

In [7]:
import json

default_params = {    
    'max_results': 5,
    'topic': "general",
    'include_answer': False,
    'include_raw_content': False,
    'include_images': False,
    'include_image_descriptions': False,
    'search_depth': "basic",
    'time_range': "",
    'include_domains': [],
    'exclude_domains': [],
    'country': "",
}
user_params = {"include_domains": ["*.it"], "exclude_domains": ["*.com", "wikipedia.org"], "country": "italy"}
params = {**default_params, **user_params}
params_json = json.dumps(params)

research_query = "\"Pasta alla carbonara\""
query = f"text: {research_query}, params: {params_json}"

result = agent.invoke(
    {"messages": [{"role": "user", "content": query}]},
    {"configurable": {"thread_id": "1"}},
)

###### NOT STRUCTURED RESULT

In [8]:
result

{'messages': [HumanMessage(content='text: "Pasta alla carbonara", params: {"max_results": 5, "topic": "general", "include_answer": false, "include_raw_content": false, "include_images": false, "include_image_descriptions": false, "search_depth": "basic", "time_range": "", "include_domains": ["*.it"], "exclude_domains": ["*.com", "wikipedia.org"], "country": "italy"}', additional_kwargs={}, response_metadata={}, id='f3a7fbaf-baf9-4237-b259-6ba172c0b2d7'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5:9b', 'created_at': '2026-04-12T13:00:24.97747Z', 'done': True, 'done_reason': 'stop', 'total_duration': 113843742100, 'load_duration': 9468577900, 'prompt_eval_count': 1459, 'prompt_eval_duration': 53987934500, 'eval_count': 168, 'eval_duration': 50160592200, 'model_name': 'qwen3.5:9b', 'model_provider': 'ollama'}, id='lc_run--019d81c5-b555-7413-ac6a-13d9b7f506c1-0', tool_calls=[{'name': 'tavily_search', 'args': {'time_range': '', 'include_domains': ['*.

### Results filtering

In [9]:
overview = result["messages"][-1]
overview.pretty_print()

================================== Ai Message ==================================

Ho trovato diverse informazioni interessanti sulla famosa **Pasta alla Carbonara** romana! 🍝

## Ingredienti Tradizionali
La carbonara è un piatto romano classico basato su pochi ingredienti essenziali:
- **Uova** (solitamente solo tuorli per la cremosità)
- **Formaggio** (Pecorino Romano o Parmigiano Reggiano)
- **Guanciale** (o pancetta)
- **Pepe nero**

## Tipi di Pasta
Sebbene lo **spaghetti** sia l'opzione tradizionale, si possono usare anche:
- Fettuccine
- Rigatoni
- Linguine
- Bucatini

## Origine della Carbonara
L'origine del nome è oggetto di diverse teorie:

1. **Carbognari**: Alcuni attribuiscono l'origine ai "carbonai" che cucinavano il "cacio e ova" nei boschi appenninici
2. **Seconda Guerra Mondiale**: Un'altra teoria data nel 1944, quando i soldati americani trovarono nelle mense alleate uova e pancetta che ricordavano il loro "breakfast" americano

3. **Evoluzione dal cacio e ova**: La ca

In [10]:
import json

tool_msg = json.loads(result["messages"][-2].content)

research_results = tool_msg["results"]
research_results

[{'url': 'https://www.italiaregina.it/carbonara-spaghetti/',
  'title': 'Carbonara Spaghetti the queen of the city of Rome',
  'content': 'carbonara spaghettiCarbonara Spaghetti is an\xa0Italian\xa0pasta\xa0dish from\xa0Rome based on\xa0eggs, cheese (Pecorino Romano\xa0or Parmigiano-Reggiano), bacon (guanciale\xa0or\xa0pancetta), and\xa0black pepper. Spaghetti\xa0is usually used as the pasta; however,\xa0fettuccine,\xa0rigatoni,\xa0linguine\xa0or\xa0bucatini\xa0can also be used. * 1 pound dry Spaghetti Pasta. * 1/2 cup freshly grated Parmigiano Reggiano. Add the spaghetti and cook for 8 to 10 minutes or until al dente. When the pasta is done, reserve 1/2 cup of the water, then drain. While the pasta is cooking, heat a large skillet over medium heat. Return the guanciale pan to medium heat, and add half of the reserved pasta water to the pan. If the sauce seems too thick, thin it out with a little bit more of the reserved pasta water. This site uses profiling cookies for different purpo

# WEB PAGE EXTRACTION INVOCATION

In [11]:
from langchain.messages import HumanMessage

query = HumanMessage(content=[
    {"type": "text", "text": [research_results[0]["url"], research_results[-1]["url"]]},
])

response = agent.invoke(
    {"messages": [query]},
    {"configurable": {"thread_id": "1"}},
)

In [12]:
# DISPLAY TOOL MESSAGE
from IPython.display import Markdown, display

display(Markdown(response["messages"][-2].content))

{"results": [{"url": "https://www.italiaregina.it/carbonara-spaghetti/", "title": "Carbonara Spaghetti the queen of the city of Rome", "raw_content": "+39 02 89054160 [info@italiaregina.it](mailto:info@italiaregina.it)\n\n* [Facebook](https://www.facebook.com/ItaliaReginashop/?ref=bookmarks)\n* [X](https://twitter.com/Italia_Regina)\n* [Instagram](https://www.instagram.com/italia_regina/)\n\n* [Facebook](https://www.facebook.com/ItaliaReginashop/?ref=bookmarks)\n* [X](https://twitter.com/Italia_Regina)\n* [Instagram](https://www.instagram.com/italia_regina/)\n\n* [Restaurants](https://business.italiaregina.it/)\n* [Retail](https://business.italiaregina.it/)\n* [Corporate Gifts](https://www.italiaregina.it/gifts-buy-online/gifts-buy-onlinecorporate-gifts-buy-online/)\n* [Ambassador](https://www.italiaregina.it/ambassador/)\n* [My account](https://www.italiaregina.it/mio-account/)\n* [Checkout](https://www.italiaregina.it/carrello/)\n* [Shop](https://www.italiaregina.it/negozio/)\n\n [0 Items](https://www.italiaregina.it/carrello/)\n\n[![Italiaregina.it](https://www.italiaregina.it/wp-content/uploads/2017/04/ItaliaRegina500x150.png)](https://www.italiaregina.it/)\n\nSelect Page\n\n# Carbonara Spaghetti\n\n![carbonara spaghetti](https://www.italiaregina.co.uk/wp-content/uploads/2017/05/Fotolia_48528638_Subscription_Monthly_M-300x300.jpg)Carbonara Spaghetti is an Italian pasta dish from Rome based on eggs, cheese (Pecorino Romano or Parmigiano-Reggiano), bacon (guanciale or pancetta), and black pepper. Spaghetti is usually used as the pasta; however, fettuccine, rigatoni, linguine or bucatini can also be used. The dish was created in the middle of the 20th century. The key to good spaghetti alla carbonara, like any good piece of cooking, and especially this one, is the quality of the ingredients.\n\n**Ingredients:**\n\n* 1 pound dry Spaghetti [Pasta](http://www.italiaregina.co.uk/pasta-buy-online/)\n* 4 large eggs, as fresh as possible\n* 8 ounces guanciale, pancetta or slab bacon, cubed\n* 1/2 cup freshly grated [Parmigiano Reggiano](http://www.parmigianoreggiano.com/english/default.aspx)\n* 1/2 cup freshly grated [Pecorino](https://italiaregina.it/italian-pecorino/) [PDO](https://en.wikipedia.org/wiki/Geographical_indications_and_traditional_specialities_in_the_European_Union)\n* Freshly cracked black pepper\n* Sea salt\n\n**Preparation:**\n\nBring about 6 quarts of generously salted water (it should taste like the ocean) to a boil. Add the spaghetti and cook for 8 to 10 minutes or until al dente. When the pasta is done, reserve 1/2 cup of the water, then drain.\n\nWhile the pasta is cooking, heat a large skillet over medium heat. Add the guanciale and sauté for about 3 minutes, or until the meat is crispy and golden and has rendered its fat. Turn off the heat.\n\nIn a small bowl whisk the eggs and [Parmigiano](https://italiaregina.it/italian-parmesan) and Pecorino cheeses until well-combined.\n\nReturn the guanciale pan to medium heat, and add half of the reserved pasta water to the pan. Toss in the spaghetti and agitate the pan over the heat for a few seconds until the bubbling subsides. Much of the water will evaporate\n\nRemove the pan from the heat and add the egg mixture and stirring quickly until the eggs thicken. The residual heat will cook the eggs but work quickly to prevent the eggs from scrambling. If the sauce seems too thick, thin it out with a little bit more of the reserved pasta water.\n\nSeason liberally with freshly cracked black pepper. Taste for seasoning; depending on the kind of pork used, it may not need any salt.\n\nDivide the pasta into bowls and serve immediately.\n\n#### Cart\n\nWith more €40.00 you can get free shipping\n\n#### Sua Maestà Balsamic Vinegar\n\n[![](data:image/svg+xml,%3Csvg%20xmlns=%22http://www.w3.org/2000/svg%22%20viewBox=%220%200%20289%20300%22%3E%3C/svg%3E)](https://www.italiaregina.it/product/balsamic-vinegar-denso-dolce-sua-maesta/)\n\n#### Parmigiano Reggiano “Solo Bruna”\n\n[![Parmesan (24 months)](data:image/svg+xml,%3Csvg%20xmlns=%22http://www.w3.org/2000/svg%22%20viewBox=%220%200%20300%20300%22%3E%3C/svg%3E)](https://www.italiaregina.it/product/parmesan-24-months-solo-bruna/)\n\n#### Laudemio Tuscany Oil\n\n[![Laudemio](data:image/svg+xml,%3Csvg%20xmlns=%22http://www.w3.org/2000/svg%22%20viewBox=%220%200%20300%20300%22%3E%3C/svg%3E \"Laudemio Tuscany Oil\")](https://www.italiaregina.it/product/laudemio-italian-extra-virgin-olive-oil/)\n\n#### Taralli bread snack\n\n[![taralli](data:image/svg+xml,%3Csvg%20xmlns=%22http://www.w3.org/2000/svg%22%20viewBox=%220%200%20296%20300%22%3E%3C/svg%3E)](https://www.italiaregina.it/product/taralli-multipack-bag-400-gr-10-x-40/)\n\n#### Marie Ange di Costa Chocol\n\n[![](data:image/svg+xml,%3Csvg%20xmlns=%22http://www.w3.org/2000/svg%22%20viewBox=%220%200%20300%20300%22%3E%3C/svg%3E)](https://www.italiaregina.it/product/oil-fragrance-tuscan-extra-virgin-olive-oil-200ml/)\n\n#### Recent news\n\n* [Balsamic Vinegar Gin Sour: a new classic cocktail!](https://www.italiaregina.it/balsamic-vinegar-gin-sour-a-new-classic-cocktail/)\n* [Balsamic Vinegar Cocktails: a unique twist on mixology](https://www.italiaregina.it/balsamic-vinegar-cocktails-a-unique-twist-on-mixology/)\n* [Pairing Grana Padano with craft beer: a flavorful adventure](https://www.italiaregina.it/pairing-grana-padano-with-craft-bbeer-a-flavorful-adventure/)\n* [Grana Padano: discover the stories of young farmers](https://www.italiaregina.it/grana-padano-youth-in-the-dairy-industry-share-their-stories/)\n* [Pairing Grana Padano with white wine: a match made in culinary heaven](https://www.italiaregina.it/pairing-grana-padano-white-wine/)\n\n€40.00 left to free shipping\n\n€40.00 left to free shipping\n\nThis site uses profiling cookies for different purposes. If you want to learn more or opt out of all or some cookies click  [here](https://www.italiaregina.it/cookies-policy/) . If you continue to browse or close this bar, you consent to the use of cookies. Following the entry of the EU Regulation 2016/679 (GDPR), we have amended, integrated and updated the section dedicated to the processing of data on privacy and cookie policy (see  [Privacy](https://www.italiaregina.it/privacy/)  and  [Cookie](https://www.italiaregina.it/cookies-policy/)  section).", "images": []}, {"url": "https://www.tumn.it/en/blog/the-origin-of-the-name-of-5-traditional-pasta-dishes-from-the-standard-alla-carbonara", "title": "The origin of the name of 5 traditional pasta dishes, from the standard alla carbonara | TUMN", "raw_content": "The first digital FoodHub of Made in Italy excellence: authentic quality only comes from here.\n\n[Italiano](/it \"English\")  [Espanol](/es \"English\")  [German](/de \"English\")  [French](/fr \"English\")\n\n##### Sign in\n\n###### Oppure\n\nSign in with Google. Opens in new tab\n\n##### Forgot password?\n\n# The origin of the name of 5 traditional pasta dishes, from the standard alla carbonara\n\n## How many times have you found yourself in front of the steaming and appetizing traditional Italian **pasta** dishes but you have not been able to understand the reason for the name? There are, in fact, many pasta recipes that have unusual names, behind which you can hide a piece of history of our country, the name of the area of origin or some singular anecdote about their origin. Let's find out what is hidden behind the names of five Italian pasta dishes that we all know.\n\n# **Carbonara pasta** (Lazio)\n\n# \n\nOn the birth of this Lazio recipe, so loved throughout the national territory as well as known and imitated also abroad, more or less verified hypotheses have followed and the riddle seems to be not yet fully revealed. According to one of the most widespread and accredited theories, carbonara was invented by the lumberjacks who went to the Apennine mountains, between Lazio and Abruzzo. In fact, it would seem that the lumberjacks, after extracting the coal from the wood, then used it to prepare a plate of pasta outdoors, using three simple ingredients they had available, namely eggs, bacon and pecorino. According to another hypothesis, a little more imaginative, the origin of [**the carbonara pasta**](https://www.lacucinaitaliana.it/ricetta/primi/spaghetti-alla-carbonara/) would derive instead from a noblewoman from Polesine who invented this recipe on the occasion of the secret Carbonari meetings that she hosted at her inn.\n\n# **Gricia pasta** (Lazio)\n\n# \n\nThis pasta dish, cooked with poor but tasty ingredients, i.e. bacon, grated pecorino romano, salt and pepper, is typical of Lazio cuisine but less known than carbonara or amatriciana. The direct ancestor of this recipe is in fact, and it is not by chance that it is also called \"the white amatriciana\".  \n  \nThe [**pasta gricia**](https://www.ricetteromane.it/pasta/pasta-alla-gricia-o-amatriciana-bianca/) was born in the '400 in a village called Grisciano Lazio, a few kilometers from amateur. The recipe seems to have been invented by local bakers called \"grici\", or immigrants from the German areas of the Rhine and in particular from the area called Canton de 'Grigioni, which would explain the name \"gricia\".  \n  \nIf the most accredited theory places precisely the invention of this pasta in the shops of these bakers, equipped with coal burners, there are those who believe that the recipe was the work of the shepherds of Grisciano, intent on grazing their flocks in the Roman countryside followed by dry pasta, bacon, lard, pecorino cheese and pepper.\n\n# **Pasta all'amatriciana** (Lazio)\n\n# \n\nThe [**bucatini all'amatriciana**](https://www.comune.amatrice.rieti.it/gli-spaghetti-allamatriciana/) , traditionally prepared with fresh tomato sauce, bacon, pecorino cheese, salt and pepper, are a very popular dish from Amatrice, a Lazio town in the province of Rieti. This town, and consequently the origin of this traditional recipe, unfortunately became known above all on the occasion of the earthquake which in August 2016 almost completely destroyed its center. As we have seen, amatriciana pasta derives from the recipe of the gricia, to which tomato sauce was simply added; in the nineteenth and twentieth centuries, some original hosts from Amatrice brought this recipe to the capital and decreed its diffusion and consolidated fame, so much so that the dish ended up being considered one of the classics of Roman cuisine.\n\n# **Pasta alla puttanesca** (Campania)\n\n# \n\nFrom Lazio we go down to Campania, and in particular to Naples, where [**spaghetti alla puttanesca**](http://www.lericetteromane.it/ricette-primi-spaghetti-alla-puttanesca) seem to have been born . This typical dish of Campania cuisine with its \"picturesque\" name, is prepared with a tomato sauce seasoned with black Gaeta olives, capers, oregano and chilli pepper. The story goes that the idea of puttanesca pasta came, in the twentieth century, to the owner of a \"pleasure house\" in the Spanish Quarters, to refresh visitors. In Naples this pasta is generally prepared with spaghetti, vermicelli or linguine and is also known as \"aulive and cchiapparielle\". Although with some variations, including the type of dry pasta used (penne or spaghetti) and the addition of anchovies in salt, the recipe soon became popular in Rome too.\n\n# **Pasta alla norma** (Sicily)\n\n# \n\nWe go further south to talk about Sicilian [**pasta alla norma**](https://www.pastaallanorma.it/) , a tasty Catania specialty famous throughout Italy. The original recipe for this pearl of Mediterranean gastronomy involves a fresh tomato sauce, fried aubergines, grated salted ricotta and fresh basil leaves, while the pasta considered most suitable is the celery rigati. The name of the dish was probably chosen by a Sicilian chef as a tribute to the most famous opera of the Catania composer Vincenzo Bellini, on the occasion of the celebrations in his hometown.\n\n*by [Ivana De Innocentis](https://www.linkedin.com/in/ivanadeinnocentis/)  \nRevised, corrected and expanded by the Tumn® editorial staff*\n\n[pasta](blog/?tags=pasta \"Show all posts with this tag\")\n\n[traditional dishes](blog/?tags=traditional dishes \"Show all posts with this tag\")\n\n[history of recipes](blog/?tags=history of recipes \"Show all posts with this tag\")\n\n#### Share\n\n* [Share](http://www.facebook.com/sharer.php?u=https://www.tumn.it/it/blog/the-origin-of-the-name-of-5-traditional-pasta-dishes-from-the-standard-alla-carbonara)\n* [Tweet](https://twitter.com/intent/tweet?url=https://www.tumn.it/it/blog/the-origin-of-the-name-of-5-traditional-pasta-dishes-from-the-standard-alla-carbonara)\n* [Pin It](javascript:void((function()%7Bvar%20e=document.createElement('script');e.setAttribute('type','text/javascript');e.setAttribute('charset','UTF-8');e.setAttribute('src','http://assets.pinterest.com/js/pinmarklet.js?r='+Math.random()*99999999);document.body.appendChild(e)%7D)());)\n* [Share](http://www.linkedin.com/shareArticle?mini=true&url=https://www.tumn.it/it/blog/the-origin-of-the-name-of-5-traditional-pasta-dishes-from-the-standard-alla-carbonara)\n\n## Products\n\n[Artisanal savory cantucci with dried tomatoes, oregano, and chili pepper - Biscotteria Bettina 150g tin](products/eat/biscuits-and-salty-snacks/artisanal-savory-cantucci-with-dried-tomatoes-oregano-and-chili-pepper-biscotteria-bettina-150g-tin \"Maggiori informazioni sul prodotto\")\n\n11,00 €\n\n[Savory tasting biscuit with aromatic herbs - Biscotteria Bettina 150 gr case.](products/eat/biscuits-and-salty-snacks/savory-tasting-biscuit-with-aromatic-herbs-biscotteria-bettina-150-gr-case \"Maggiori informazioni sul prodotto\")\n\n6,50 €\n\n[Chocolate and hazelnut biscuits - Biscotteria Bettina, 150g tin.](products/eat/sweets-and-biscuits/chocolate-and-hazelnut-biscuits-biscotteria-bettina-150g-tin \"Maggiori informazioni sul prodotto\")\n\n12,10 €\n\n[Aztec Chocolate Powder for Hot Chocolate - Marco Colzani 180g pack](products/eat/sweets-and-biscuits/aztec-chocolate-powder-for-hot-chocolate-marco-colzani-180g-pack \"Maggiori informazioni sul prodotto\")\n\n9,00 €\n\n[Attilio Servi Artisan Gourmet Panettone 750g - Natural Sourdough with Calabrian Orange and Extra Jumbo Raisins - Award-Winning Master Leavener](products/sweets-and-biscuits/attilio-servi-artisan-gourmet-panettone-750g-natural-sourdough-with-calabrian-orange-and-extra-jumbo-raisins-award-winning-master-leavener \"Maggiori informazioni sul prodotto\")\n\n33,00 €\n\n[Borettane Onions - Designer Garden, 270g pack.](products/eat/preserves-and-pickles/borettane-onions-designer-garden-270g-pack \"Maggiori informazioni sul prodotto\")\n\n9,00 €\n\nhere\n\n[Artisanal savory cantucci with dried tomatoes, oregano, and chili pepper - Biscotteria Bettina 150g tin](products/eat/biscuits-and-salty-snacks/artisanal-savory-cantucci-with-dried-tomatoes-oregano-and-chili-pepper-biscotteria-bettina-150g-tin \"Maggiori informazioni sul prodotto\")\n\n11,00 €\n\n[Savory tasting biscuit with aromatic herbs - Biscotteria Bettina 150 gr case.](products/eat/biscuits-and-salty-snacks/savory-tasting-biscuit-with-aromatic-herbs-biscotteria-bettina-150-gr-case \"Maggiori informazioni sul prodotto\")\n\n6,50 €\n\n[Chocolate and hazelnut biscuits - Biscotteria Bettina, 150g tin.](products/eat/sweets-and-biscuits/chocolate-and-hazelnut-biscuits-biscotteria-bettina-150g-tin \"Maggiori informazioni sul prodotto\")\n\n12,10 €\n\n[Aztec Chocolate Powder for Hot Chocolate - Marco Colzani 180g pack](products/eat/sweets-and-biscuits/aztec-chocolate-powder-for-hot-chocolate-marco-colzani-180g-pack \"Maggiori informazioni sul prodotto\")\n\n9,00 €\n\n[Attilio Servi Artisan Gourmet Panettone 750g - Natural Sourdough with Calabrian Orange and Extra Jumbo Raisins - Award-Winning Master Leavener](products/sweets-and-biscuits/attilio-servi-artisan-gourmet-panettone-750g-natural-sourdough-with-calabrian-orange-and-extra-jumbo-raisins-award-winning-master-leavener \"Maggiori informazioni sul prodotto\")\n\n33,00 €\n\n[Borettane Onions - Designer Garden, 270g pack.](products/eat/preserves-and-pickles/borettane-onions-designer-garden-270g-pack \"Maggiori informazioni sul prodotto\")\n\n9,00 €", "images": []}], "failed_results": [], "response_time": 0.01, "request_id": "03f96cea-e0ad-4fb3-bb60-bf1c2618f50c"}

In [13]:
# DISPLAY AI MESSAGE
from IPython.display import Markdown, display

display(Markdown(response["messages"][-1].content))

----- SOURCE: https://www.italiaregina.it/carbonara-spaghetti/
+39 02 89054160 [info@italiaregina.it](mailto:info@italiaregina.it)

* [Facebook](https://www.facebook.com/ItaliaReginashop/?ref=bookmarks)
* [X](https://twitter.com/Italia_Regina)
* [Instagram](https://www.instagram.com/italia_regina/)

* [Facebook](https://www.facebook.com/ItaliaReginashop/?ref=bookmarks)
* [X](https://twitter.com/Italia_Regina)
* [Instagram](https://www.instagram.com/italia_regina/)

* [Restaurants](https://business.italiaregina.it/)
* [Retail](https://business.italiaregina.it/)
* [Corporate Gifts](https://www.italiaregina.it/gifts-buy-online/gifts-buy-onlinecorporate-gifts-buy-online/)
* [Ambassador](https://www.italiaregina.it/ambassador/)
* [My account](https://www.italiaregina.it/mio-account/)
* [Checkout](https://www.italiaregina.it/carrello/)
* [Shop](https://www.italiaregina.it/negozio/)

 [0 Items](https://www.italiaregina.it/carrello/)

[![Italiaregina.it](https://www.italiaregina.it/wp-content/uploads/2017/04/ItaliaRegina500x150.png)](https://www.italiaregina.it/)

Select Page

# Carbonara Spaghetti

![carbonara spaghetti](https://www.italiaregina.co.uk/wp-content/uploads/2017/05/Fotolia_48528638_Subscription_Monthly_M-300x300.jpg)Carbonara Spaghetti is an  Italian  pasta  dish from  Rome based on  eggs, cheese (Pecorino Romano  or Parmigiano-Reggiano), bacon (guanciale  or pancetta), and  black pepper.  Spaghetti  is usually used as the pasta; however,  fettuccine,  rigatoni,  linguine  or  bucatini  can also be used. The dish was created in the middle of the 20th century. The key to good spaghetti alla carbonara, like any good piece of cooking, and especially this one, is the quality of the ingredients.

**Ingredients:**

* 1 pound dry Spaghetti [Pasta](http://www.italiaregina.co.uk/pasta-buy-online/)
* 4 large eggs, as fresh as possible
* 8 ounces guanciale, pancetta or slab bacon, cubed
* 1/2 cup freshly grated [Parmigiano Reggiano](http://www.parmigianoreggiano.com/english/default.aspx)
* 1/2 cup freshly grated [Pecorino](https://italiaregina.it/italian-pecorino/)  [PDO](https://en.wikipedia.org/wiki/Geographical_indications_and_traditional_specialities_in_the_European_Union)
* Freshly cracked black pepper
* Sea salt

**Preparation:**

Bring about 6 quarts of generously salted water (it should taste like the ocean) to a boil. Add the spaghetti and cook for 8 to 10 minutes or until al dente. When the pasta is done, reserve 1/2 cup of the water, then drain.

While the pasta is cooking, heat a large skillet over medium heat. Add the guanciale and sauté for about 3 minutes, or until the meat is crispy and golden and has rendered its fat. Turn off the heat.

In a small bowl whisk the eggs and [Parmigiano](https://italiaregina.it/italian-parmesan) and Pecorino cheeses until well-combined.

Return the guanciale pan to medium heat, and add half of the reserved pasta water to the pan. Toss in the spaghetti and agitate the pan over the heat for a few seconds until the bubbling subsides. Much of the water will evaporate

Remove the pan from the heat and add the egg mixture and stirring quickly until the eggs thicken. The residual heat will cook the eggs but work quickly to prevent the eggs from scrambling. If the sauce seems too thick, thin it out with a little bit more of the reserved pasta water.

Season liberally with freshly cracked black pepper. Taste for seasoning; depending on the kind of pork used, it may not need any salt.

Divide the pasta into bowls and serve immediately.

#### Cart

With more €40.00 you can get free shipping

#### Sua Maestà Balsamic Vinegar

[![](data:image/svg+xml,%3Csvg%20xmlns=%22http://www.w3.org/2000/svg%22%20viewBox=%220%200%20289%20300%22%3E%3C/svg%3E)](https://www.italiaregina.it/product/balsamic-vinegar-denso-dolce-sua-maesta/)

#### Parmigiano Reggiano "Solo Bruna"

[![Parmesan (24 months)](data:image/svg+xml,%3Csvg%20xmlns=%22http://www.w3.org/2000/svg%22%20viewBox=%220%200%20300%20300%22%3E%3C/svg%3E)](https://www.italiaregina.it/product/parmesan-24-months-solo-bruna/)

#### Laudemio Tuscany Oil

[![Laudemio](data:image/svg+xml,%3Csvg%20xmlns=%22http://www.w3.org/2000/svg%22%20viewBox=%220%200%20300%20300%22%3E%3C/svg%3E "Laudemio Tuscany Oil")](https://www.italiaregina.it/product/laudemio-italian-extra-virgin-olive-oil/)

#### Taralli bread snack

[![taralli](data:image/svg+xml,%3Csvg%20xmlns=%22http://www.w3.org/2000/svg%22%20viewBox=%220%200%20296%20300%22%3E%3C/svg%3E)](https://www.italiaregina.it/product/taralli-multipack-bag-400-gr-10-x-40/)

#### Marie Ange di Costa Chocol

[![](data:image/svg+xml,%3Csvg%20xmlns=%22http://www.w3.org/2000/svg%22%20viewBox=%220%200%20300%20300%22%3E%3C/svg%3E)](https://www.italiaregina.it/product/oil-fragrance-tuscan-extra-virgin-olive-oil-200ml/)

#### Recent news

* [Balsamic Vinegar Gin Sour: a new classic cocktail!](https://www.italiaregina.it/balsamic-vinegar-gin-sour-a-new-classic-cocktail/)
* [Balsamic Vinegar Cocktails: a unique twist on mixology](https://www.italiaregina.it/balsamic-vinegar-cocktails-a-unique-twist-on-mixology/)
* [Pairing Grana Padano with craft beer: a flavorful adventure](https://www.italiaregina.it/pairing-grana-padano-with-craft-beer-a-flavorful-adventure/)
* [Grana Padano: discover the stories of young farmers](https://www.italiaregina.it/grana-padano-youth-in-the-dairy-industry-share-their-stories/)
* [Pairing Grana Padano with white wine: a match made in culinary heaven](https://www.italiaregina.it/pairing-grana-padano-white-wine/)

€40.00 left to free shipping

€40.00 left to free shipping

This site uses profiling cookies for different purposes. If you want to learn more or opt out of all or some cookies click  [here](https://www.italiaregina.it/cookies-policy/) . If you continue to browse or close this bar, you consent to the use of cookies. Following the entry of the EU Regulation 2016/679 (GDPR), we have amended, integrated and updated the section dedicated to the processing of data on privacy and cookie policy (see  [Privacy](https://www.italiaregina.it/privacy/)  and  [Cookie](https://www.italiaregina.it/cookies-policy/)  section).



----- SOURCE: https://www.tumn.it/en/blog/the-origin-of-the-name-of-5-traditional-pasta-dishes-from-the-standard-alla-carbonara
The first digital FoodHub of Made in Italy excellence: authentic quality only comes from here.

[Italiano](/it "English")  [Espanol](/es "English")  [German](/de "English")  [French](/fr "English")

##### Sign in

###### Oppure

Sign in with Google. Opens in new tab

##### Forgot password?

# The origin of the name of 5 traditional pasta dishes, from the standard alla carbonara

## How many times have you found yourself in front of the steaming and appetizing traditional Italian **pasta** dishes but you have not been able to understand the reason for the name? There are, in fact, many pasta recipes that have unusual names, behind which you can hide a piece of history of our country, the name of the area of origin or some singular anecdote about their origin. Let's find out what is hidden behind the names of five Italian pasta dishes that we all know.

# **Carbonara pasta** (Lazio)

# 

On the birth of this Lazio recipe, so loved throughout the national territory as well as known and imitated also abroad, more or less verified hypotheses have followed and the riddle seems to be not yet fully revealed. According to one of the most widespread and accredited theories, carbonara was invented by the lumberjacks who went to the Apennine mountains, between Lazio and Abruzzo. In fact, it would seem that the lumberjacks, after extracting the coal from the wood, then used it to prepare a plate of pasta outdoors, using three simple ingredients they had available, namely eggs, bacon and pecorino. According to another hypothesis, a little more imaginative, the origin of [**the carbonara pasta**](https://www.lacucinaitaliana.it/ricetta/primi/spaghetti-alla-carbonara/) would derive instead from a noblewoman from Polesine who invented this recipe on the occasion of the secret Carbonari meetings that she hosted at her inn.

# **Gricia pasta** (Lazio)

# 

This pasta dish, cooked with poor but tasty ingredients, i.e. bacon, grated pecorino romano, salt and pepper, is typical of Lazio cuisine but less known than carbonara or amatriciana. The direct ancestor of this recipe is in fact, and it is not by chance that it is also called "the white amatriciana".  

The [**pasta gricia**](https://www.ricetteromane.it/pasta/pasta-alla-gricia-o-amatriciana-bianca/) was born in the '400 in a village called Grisciano Lazio, a few kilometers from amateur. The recipe seems to have been invented by local bakers called "grici", or immigrants from the German areas of the Rhine and in particular from the area called Canton de 'Grigioni, which would explain the name "gricia".  

If the most accredited theory places precisely the invention of this pasta in the shops of these bakers, equipped with coal burners, there are those who believe that the recipe was the work of the shepherds of Grisciano, intent on grazing their flocks in the Roman countryside followed by dry pasta, bacon, lard, pecorino cheese and pepper.

# **Pasta all'amatriciana** (Lazio)

# 

The [**bucatini all'amatriciana**](https://www.comune.amatrice.rieti.it/gli-spaghetti-allamatriciana/) , traditionally prepared with fresh tomato sauce, bacon, pecorino cheese, salt and pepper, are a very popular dish from Amatrice, a Lazio town in the province of Rieti. This town, and consequently the origin of this traditional recipe, unfortunately became known above all on the occasion of the earthquake which in August 2016 almost completely destroyed its center. As we have seen, amatriciana pasta derives from the recipe of the gricia, to which tomato sauce was simply added; in the nineteenth and twentieth centuries, some original hosts from Amatrice brought this recipe to the capital and decreed its diffusion and consolidated fame, so much so that the dish ended up being considered one of the classics of Roman cuisine.

# **Pasta alla puttanesca** (Campania)

# 

From Lazio we go down to Campania, and in particular to Naples, where [**spaghetti alla puttanesca**](http://www.lericetteromane.it/ricette-primi-spaghetti-alla-puttanesca) seem to have been born . This typical dish of Campania cuisine with its "picturesque" name, is prepared with a tomato sauce seasoned with black Gaeta olives, capers, oregano and chilli pepper. The story goes that the idea of puttanesca pasta came, in the twentieth century, to the owner of a "pleasure house" in the Spanish Quarters, to refresh visitors. In Naples this pasta is generally prepared with spaghetti, vermicelli or linguine and is also known as "aulive and cchiapparielle". Although with some variations, including the type of dry pasta used (penne or spaghetti) and the addition of anchovies in salt, the recipe soon became popular in Rome too.

# **Pasta alla norma** (Sicily)

# 

We go further south to talk about Sicilian [**pasta alla norma**](https://www.pastaallanorma.it/) , a tasty Catania specialty famous throughout Italy. The original recipe for this pearl of Mediterranean gastronomy involves a fresh tomato sauce, fried aubergines, grated salted ricotta and fresh basil leaves, while the pasta considered most suitable is the celery rigati. The name of the dish was probably chosen by a Sicilian chef as a tribute to the most famous opera of the Catania composer Vincenzo Bellini, on the occasion of the celebrations in his hometown.

*by [Ivana De Innocentis](https://www.linkedin.com/in/ivanadeinnocentis/)  
Revised, corrected and expanded by the Tumn® editorial staff*

[pasta](blog/?tags=pasta "Show all posts with this tag")

[traditional dishes](blog/?tags=traditional dishes "Show all posts with this tag")

[history of recipes](blog/?tags=history of recipes "Show all posts with this tag")

#### Share

* [Share](http://www.facebook.com/sharer.php?u=https://www.tumn.it/it/blog/the-origin-of-the-name-of-5-traditional-pasta-dishes-from-the-standard-alla-carbonara)
* [Tweet](https://twitter.com/intent/tweet?url=https://www.tumn.it/it/blog/the-origin-of-the-name-of-5-traditional-pasta-dishes-from-the-standard-alla-carbonara)
* [Pin It](javascript:void((function()%7Bvar%20e=document.createElement('script');e.setAttribute('type','text/javascript');e.setAttribute('charset','UTF-8');e.setAttribute('src','http://assets.pinterest.com/js/pinmarklet.js?r='+Math.random()*99999999);document.body.appendChild(e)%7D)());)
* [Share](http://www.linkedin.com/shareArticle?mini=true&url=https://www.tumn.it/it/blog/the-origin-of-the-name-of-5-traditional-pasta-dishes-from-the-standard-alla-carbonara)

## Products

[Artisanal savory cantucci with dried tomatoes, oregano, and chili pepper - Biscotteria Bettina 150g tin](products/eat/biscuits-and-salty-snacks/artisanal-savory-cantucci-with-dried-tomatoes-oregano-and-chili-pepper-biscotteria-bettina-150g-tin "Maggiori informazioni sul prodotto")

11,00 €

[Savory tasting biscuit with aromatic herbs - Biscotteria Bettina 150 gr case.](products/eat/biscuits-and-salty-snacks/savory-tasting-biscuit-with-aromatic-herbs-biscotteria-bettina-150-gr-case "Maggiori informazioni sul prodotto")

6,50 €

[Chocolate and hazelnut biscuits - Biscotteria Bettina, 150g tin.](products/eat/sweets-and-biscuits/chocolate-and-hazelnut-biscuits-biscotteria-bettina-150g-tin "Maggiori informazioni sul prodotto")

12,10 €

[Aztec Chocolate Powder for Hot Chocolate - Marco Colzani 180g pack](products/eat/sweets-and-biscuits/aztec-chocolate-powder-for-hot-chocolate-marco-colzani-180g-pack "Maggiori informazioni sul prodotto")

9,00 €

[Attilio Servi Artisan Gourmet Panettone 750g - Natural Sourdough with Calabrian Orange and Extra Jumbo Raisins - Award-Winning Master Leavener](products/sweets-and-biscuits/attilio-servi-artisan-gourmet-panettone-750g-natural-sourdough-with-calabrian-orange-and-extra-jumbo-raisins-award-winning-master-leavener "Maggiori informazioni sul prodotto")

33,00 €

[Borettane Onions - Designer Garden, 270g pack.](products/eat/preserves-and-pickles/borettane-onions-designer-garden-270g-pack "Maggiori informazioni sul prodotto")

9,00 €

here

[Artisanal savory cantucci with dried tomatoes, oregano, and chili pepper - Biscotteria Bettina 150g tin](products/eat/biscuits-and-salty-snacks/artisanal-savory-cantucci-with-dried-tomatoes-oregano-and-chili-pepper-biscotteria-bettina-150g-tin "Maggiori informazioni sul prodotto")

11,00 €

[Savory tasting biscuit with aromatic herbs - Biscotteria Bettina 150 gr case.](products/eat/biscuits-and-salty-snacks/savory-tasting-biscuit-with-aromatic-herbs-biscotteria-bettina-150-gr-case "Maggiori informazioni sul prodotto")

6,50 €

[Chocolate and hazelnut biscuits - Biscotteria Bettina, 150g tin.](products/eat/sweets-and-biscuits/chocolate-and-hazelnut-biscuits-biscotteria-bettina-150g-tin "Maggiori informazioni sul prodotto")

12,10 €

[Aztec Chocolate Powder for Hot Chocolate - Marco Colzani 180g pack](products/eat/sweets-and-biscuits/aztec-chocolate-powder-for-hot-chocolate-marco-colzani-180g-pack "Maggiori informazioni sul prodotto")

9,00 €

[Attilio Servi Artisan Gourmet Panettone 750g - Natural Sourdough with Calabrian Orange and Extra Jumbo Raisins - Award-Winning Master Leavener](products/sweets-and-biscuits/attilio-servi-artisan-gourmet-panettone-750g-natural-sourdough-with-calabrian-orange-and-extra-jumbo-raisins-award-winning-master-leavener "Maggiori informazioni sul prodotto")

33,00 €

[Borettane Onions - Designer Garden, 270g pack.](products/eat/preserves-and-pickles/borettane-onions-designer-garden-270g-pack "Maggiori informazioni sul prodotto")

9,00 €

# DOCUMENT LOADING

In [14]:
import json
from langchain_core.documents import Document

tool_results = json.loads(response["messages"][-2].content).get("results")

docs=[]
for result in tool_results:
    docs.append(
        Document(
            page_content = result.get("raw_content"),
            metadata = {"url": result.get("url"), "title": result.get("title")},
        )
    )

###### ONLY FOR DEBUG PURPOSES

In [15]:
for doc in docs:
    print(doc.metadata["url"])
    print(f"{doc.page_content[:400]}\n\n\n\n")    

https://www.italiaregina.it/carbonara-spaghetti/
+39 02 89054160 [info@italiaregina.it](mailto:info@italiaregina.it)

* [Facebook](https://www.facebook.com/ItaliaReginashop/?ref=bookmarks)
* [X](https://twitter.com/Italia_Regina)
* [Instagram](https://www.instagram.com/italia_regina/)

* [Facebook](https://www.facebook.com/ItaliaReginashop/?ref=bookmarks)
* [X](https://twitter.com/Italia_Regina)
* [Instagram](https://www.instagram.com/italia_reg




https://www.tumn.it/en/blog/the-origin-of-the-name-of-5-traditional-pasta-dishes-from-the-standard-alla-carbonara
The first digital FoodHub of Made in Italy excellence: authentic quality only comes from here.

[Italiano](/it "English")  [Espanol](/es "English")  [German](/de "English")  [French](/fr "English")

##### Sign in

###### Oppure

Sign in with Google. Opens in new tab

##### Forgot password?

# The origin of the name of 5 traditional pasta dishes, from the standard alla carbonara

## How many times 






# DOCUMENT SPLITTING
[https://docs.langchain.com/oss/python/integrations/splitters/markdown_header_metadata_splitter]

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=120,
    separators=[r"\n## ", r"\n### ", r"\n#### ", r"\n\n", r"\n", r"(?<=\. )", " ", ""], 
    is_separator_regex=True
)

raw_splits = r_splitter.split_documents(docs)

splits = []
for split in raw_splits:
    allowed_meta = ["url", "title"]
    meta_parts = []
    for key in allowed_meta:
        if key in split.metadata:
            value = split.metadata[key]
            meta_parts.append(f"{key}: {value}")
    
    metadata_header = f"[{', '.join(meta_parts)}]\n"
    split.page_content = metadata_header + split.page_content
    splits.append(split)    

###### ONLY FOR DEBUG PURPOSES

In [17]:
_ = [print(split.metadata["url"] + "\n" + split.metadata["title"] + "\n" + split.page_content + "\n\n") for split in splits]

https://www.italiaregina.it/carbonara-spaghetti/
Carbonara Spaghetti the queen of the city of Rome
[url: https://www.italiaregina.it/carbonara-spaghetti/, title: Carbonara Spaghetti the queen of the city of Rome]
+39 02 89054160 [info@italiaregina.it](mailto:info@italiaregina.it)

* [Facebook](https://www.facebook.com/ItaliaReginashop/?ref=bookmarks)
* [X](https://twitter.com/Italia_Regina)
* [Instagram](https://www.instagram.com/italia_regina/)

* [Facebook](https://www.facebook.com/ItaliaReginashop/?ref=bookmarks)
* [X](https://twitter.com/Italia_Regina)
* [Instagram](https://www.instagram.com/italia_regina/)


https://www.italiaregina.it/carbonara-spaghetti/
Carbonara Spaghetti the queen of the city of Rome
[url: https://www.italiaregina.it/carbonara-spaghetti/, title: Carbonara Spaghetti the queen of the city of Rome]
* [Restaurants](https://business.italiaregina.it/)
* [Retail](https://business.italiaregina.it/)
* [Corporate Gifts](https://www.italiaregina.it/gifts-buy-online/gift

# DOCUMENT EMBEDDINGS

In [18]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import os

current_directory = os.getcwd()

ids = ["id" + str(index + 1) for index, val in enumerate(splits)]

vector_store = Chroma(
    collection_name="webpages_content_collection",
    embedding_function=HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2"),
    #persist_directory=current_directory, 
)

vector_store.add_documents(
    documents = splits,
    #ids = ids,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['id1',
 'id2',
 'id3',
 'id4',
 'id5',
 'id6',
 'id7',
 'id8',
 'id9',
 'id10',
 'id11',
 'id12',
 'id13',
 'id14',
 'id15',
 'id16',
 'id17',
 'id18',
 'id19',
 'id20',
 'id21',
 'id22',
 'id23',
 'id24',
 'id25',
 'id26',
 'id27',
 'id28',
 'id29',
 'id30',
 'id31',
 'id32',
 'id33',
 'id34',
 'id35',
 'id36',
 'id37',
 'id38',
 'id39',
 'id40',
 'id41',
 'id42',
 'id43']

# RAG-BASED CONVERSATION

In [19]:
query = ("Quanto pepe devo macinare per fare una carbonara perfetta?")

result = agent.invoke(
    {"messages": [{"role": "user", "content": query}]},
    {"configurable": {"thread_id": "1"}},
)

result["messages"][-1].pretty_print()

================================== Ai Message ==================================

Sulla base delle fonti che ho recuperato, per una carbonara perfetta:

## Quantità di Pepe Nero

Le fonti indicano di usare **"pepe nero macinato fresco"** (freshly cracked black pepper) e di **condirne generosamente** (season liberally).

Non viene specificato una quantità esatta in grammi o cucchiaini, ma il consiglio è di **condire liberamente** con il pepe macinato al momento, a seconda del tuo gusto personale.

## Consigli dalle Fonti

1. **Macina il pepe al momento**: Usa sempre pepe macinato fresco, non pre-grattato
2. **Adatta alla quantità**: "Taste for seasoning; depending on the kind of pork used, it may not need any salt" - prova il sapore prima di aggiungere altro pepe
3. **Non saltare il pepe**: È considerato un ingrediente essenziale del piatto

## Attenzione

Una delle fonti specifica anche: **"Use less salt as usual because pasta have already a tasty condiment with cheek lard, parmesan an

In [20]:
query = ("Quanto marsala è richiesto?")

result = agent.invoke(
    {"messages": [{"role": "user", "content": query}]},
    {"configurable": {"thread_id": "1"}},
)

result["messages"][-1].pretty_print()

================================== Ai Message ==================================

Sembra ci sia un malinteso! **Il Marsala NON è un ingrediente della carbonara autentica.** 🍷❌

## Ingredienti Autentici della Carbonara

Secondo le fonti che ho recuperato, gli ingredienti tradizionali sono solo:

1. **Uova** (tuorli freschi)
2. **Guanciale** (o pancetta)
3. **Pecorino Romano** (o Parmigiano Reggiano)
4. **Pepe nero** macinato fresco
5. **Sale** (a discrezione)

## Cosa NON fa parte della Carbonara:

- ❌ **Marsala** o altro vino
- ❌ Pomodoro
- ❌ Panketto
- ❌ Olio extravergine d'oliva
- ❌ Olio (tranne il grasso del guanciale)

## Il Malinteso

Il Marsala è tipicamente usato in altri piatti italiani, come:
- Risotto al Marsala
- Osso buco al Marsala
- Torta caprese
- Deserti

Ma **non nella carbonara**!

La carbonara è celebre proprio per la sua semplicità: solo 4-5 ingredienti principali, senza vino, senza pomodoro, solo l'essenza del guanciale, delle uova e del formaggio. 🍝

Quindi, per u

In [21]:
query = ("Mi sono confuso con il marsala. Tornando al pepe, quali fonti hai consultato? Scrivimi sia i titoli sia gli url")

result = agent.invoke(
    {"messages": [{"role": "user", "content": query}]},
    {"configurable": {"thread_id": "1"}},
)

result["messages"][-1].pretty_print()

================================== Ai Message ==================================

Ho consultato **due fonti** che ho estratto direttamente per le informazioni sulla carbonara. Ecco i dettagli:

---

## 📚 Fonti Consultate

### 1. **Carbonara Spaghetti the queen of the city of Rome**
- **URL:** `https://www.italiaregina.it/carbonara-spaghetti/`
- **Contenuto relativo al pepe:**
  - Ingredienti: "Freshly cracked black pepper" (pepe nero macinato fresco)
  - Preparazione: "Season liberally with freshly cracked black pepper" (condire liberamente con pepe nero macinato fresco)

### 2. **The origin of the name of 5 traditional pasta dishes, from the standard alla carbonara**
- **URL:** `https://www.tumn.it/en/blog/the-origin-of-the-name-of-5-traditional-pasta-dishes-from-the-standard-alla-carbonara`
- **Contenuto relativo al pepe:**
  - Menziona che la carbonara si prepara con ingredienti "poor but tasty ingredients, i.e. bacon, grated pecorino romano, salt and pepper" (pancia sgrassata, peco

# ELAPSED TIME

In [ ]:
stop_time = time.perf_counter()

elapsed_time = stop_time - start_time

In [23]:
minutes = int(elapsed_time // 60)
seconds = elapsed_time % 60

print(f"Execution time: {minutes:02d}:{seconds:05.2f}")

Execution time: 61:01.33
